In [ ]:
!pip install transformers==4.48.0

In [ ]:
from pathlib import Path
import json
import torch
from transformers import pipeline

INPUT_FOLDER = Path(r'/data/demo/audio')
OUTPUT_FILE = Path(r'/content/drive/MyDrive/transcription/transcription_demo.json')

print("Input folder:", INPUT_FOLDER.resolve())
print("Output file:", OUTPUT_FILE.resolve())

Input folder: /content/drive/MyDrive/demo/audio
Output file: /content/drive/MyDrive/transcription_demo_299.json


In [ ]:
# Load the model
model_id = "Qualcomm-AI-Research/PhoASR-whisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create pipeline for automatic speech recognition with word-level timestamps
pipe = pipeline("automatic-speech-recognition", model=model_id, chunk_length_s=30, device=device, return_timestamps="word", generate_kwargs={"language": "vi"})

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda


In [5]:
BATCH_SIZE = 32

AUDIO_EXTENSIONS = {
    ".wav",
    ".mp3",
    ".m4a",
    ".flac",
    ".ogg",
    ".webm",
}


# --------------------------------------------------
# Load existing results
# --------------------------------------------------

if OUTPUT_FILE.exists():
    with OUTPUT_FILE.open("r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []


# --------------------------------------------------
# Find already processed files
# --------------------------------------------------

processed_files = {
    item["file"]
    for item in results
    if "file" in item
    and "transcription" in item
}


# --------------------------------------------------
# Find audio files
# --------------------------------------------------

audio_files = sorted(
    file
    for file in INPUT_FOLDER.iterdir()
    if file.is_file()
    and file.suffix.lower() in AUDIO_EXTENSIONS
)

# Remove files already processed
remaining_files = [
    file
    for file in audio_files
    if file.name not in processed_files
]


print(f"Total audio files: {len(audio_files)}")
print(f"Already processed: {len(processed_files)}")
print(f"Remaining: {len(remaining_files)}")
print(f"Batch size: {BATCH_SIZE}")


# --------------------------------------------------
# Process in batches
# --------------------------------------------------

for batch_start in range(0, len(remaining_files), BATCH_SIZE):

    batch = remaining_files[
        batch_start : batch_start + BATCH_SIZE
    ]

    batch_number = batch_start // BATCH_SIZE + 1
    total_batches = (
        len(remaining_files) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    print(
        f"\nBatch {batch_number}/{total_batches} "
        f"({len(batch)} files)"
    )

    try:

        # ------------------------------------------
        # RUN WHISPER INFERENCE
        # ------------------------------------------

        outputs = pipe(
            [str(file) for file in batch],
            batch_size=BATCH_SIZE,
            return_timestamps=False,
        )


        # ------------------------------------------
        # Store results
        # ------------------------------------------

        for audio_file, output in zip(batch, outputs):

            transcription = output["text"].strip()

            result = {
                "file": audio_file.name,
                "path": str(audio_file),
                "transcription": transcription,
            }

            results.append(result)

            print(
                f"  ✓ {audio_file.name}: "
                f"{transcription[:100]}"
            )


        # ------------------------------------------
        # SAVE AFTER EVERY BATCH
        # ------------------------------------------

        with OUTPUT_FILE.open("w", encoding="utf-8") as f:
            json.dump(
                results,
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(
            f"Saved {len(results)} total results."
        )


    except Exception as e:

        print(f"ERROR processing batch: {e}")
        print("Stopping so the failed batch can be investigated.")

        break


print("\nFinished.")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Total audio files: 28
Already processed: 0
Remaining: 28
Batch size: 32

Batch 1/1 (28 files)


/usr/local/lib/python3.13/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


  ✓ ctx_00aa19e0.mp3: Thành phần của mômen lưỡng cực từ của sao Hải Vương tại xích đạo từ bằng mười bốn micro tét la không
  ✓ ctx_00ada008.mp3: Từ khi có sự thành lập giải bóng đá chuyên nghiệp Nhật Bản năm một nghìn chín trăm chín mươi hai, mô
  ✓ ctx_00cde671.mp3: Hoa Anh Tơn là một thành viên của Giáo hội Anh giáo, sau này gọi là Giáo hội Giám nhiệm, có địa vị đ
  ✓ ctx_00df75b1.mp3: Năm hai nghìn không trăm linh ba, Hoa Kỳ mở một cuộc tấn công xâm chiếm I rắc dẫn đến sự sụp đổ của 
  ✓ ctx_00e923cc.mp3: Nghiên cứu nhằm cải thiện các điều trị đang có bao gồm giảm tác dụng phụ của thuốc. Đơn giản hóa phá
  ✓ ctx_00e926ac.mp3: Dựa theo hệ thống ghi chép dân số, dựa theo địa chỉ của Thổ Nhĩ Kỳ, dân số toàn quốc là bảy mươi bốn
  ✓ ctx_00f05fe8.mp3: Bồ Đào Nha có một số lễ hội âm nhạc mùa hè, như Festival Sudo S tại Zambur Dj de Jorma, Festival The
  ✓ ctx_0a12e359.mp3: Những nhà nghiên cứu kết luận rằng bệnh giang mai đã bị mang từ tân thế giới sang châu Âu sau chuyến
  ✓ ctx_0a29fa14